# HW14 Problem 3 — OpenMC $k_\infty$ for Graphite + 2% Enriched U at $R=1000$

This solution adapts the Lesson 12 model: a homogeneous material in a reflective
1 cm sphere, with reaction-rate tallies divided by flux to give effective cross
sections. Only three things change:

1. The material is Problem 2's mixture: graphite at 2.26 g/cm³ (with `c_Graphite`
   thermal scattering) plus uranium enriched to 2 atom-% U-235, at
   $N_C/N_U = 1000$.
2. The run mode is $k$-eigenvalue instead of fixed source.
3. The tallies use the three FNRP Table 3.2 groups, so that OpenMC's group fluxes
   and group constants compare directly with the analytic model from Problems 1–2.

Use a kernel that has OpenMC, NumPy, and pandas. `OPENMC_CROSS_SECTIONS` must name
a library with `C12`, `U235`, `U238`, and `c_Graphite` (ENDF/B-VIII.1 here). The
run's XML/HDF5 files go to a temporary directory.

In [1]:
import tempfile

import numpy as np
import pandas as pd
import openmc

# GLOBAL CONSTANTS
TEMPERATURE = 294.0
RADIUS = 1.0  # cm, same reflective sphere as Lesson 12
R = 1000.0  # N_C / N_U
ENRICHMENT = 0.02  # U-235 atom fraction in U
N_C = 2.26 / 12.0 * 6.02214076e23 * 1e-24  # graphite, atoms/(b cm)
N = {"C12": N_C,
     "U235": ENRICHMENT * N_C / R,
     "U238": (1.0 - ENRICHMENT) * N_C / R}
# FNRP Table 3.2 groups, ascending as OpenMC requires: g = 3 (E < 1 eV), 2, 1.
# Group 1 runs to 20 MeV so that it holds the whole fission spectrum.
GROUP_EDGES = [1.0e-5, 1.0, 1.0e5, 2.0e7]

pd.options.display.float_format = "{:.4g}".format

## 1. Model and three-group tallies

Every tally carries the three-group `EnergyFilter`. With the group flux $\phi_g$,

$$\Sigma_{x,g}=\frac{R_{x,g}}{\phi_g},\qquad
\sigma^{(i)}_{x,g}=\frac{R^{(i)}_{x,g}}{N^{(i)}\phi_g},\qquad
\Sigma_{s,g'\leftarrow g}=\frac{R_{s,g\to g'}}{\phi_g}.$$

OpenMC's nuclide-wise rates are $R^{(i)}_{x,g}=N^{(i)}\sigma^{(i)}_{x,g}\phi_g$, and
`absorption` is capture plus fission, as in the analytic $\sigma_a$. The transfer
rates $R_{s,g\to g'}$ need the outgoing energy, so the `scatter` tally with an
`EnergyoutFilter` is scored with an analog estimator.

In [2]:
def make_model(particles=5000, batches=60, inactive=10, seed=14003):
    """Lesson 12's reflective sphere in eigenvalue mode with three-group tallies."""
    material = openmc.Material(name="graphite + 2% enriched U")
    for nuclide, density in N.items():
        material.add_nuclide(nuclide, density)
    material.set_density("sum")  # the densities above are atoms/(b cm)
    material.temperature = TEMPERATURE
    material.add_s_alpha_beta("c_Graphite")

    sphere = openmc.Sphere(r=RADIUS, boundary_type="reflective")
    cell = openmc.Cell(fill=material, region=-sphere)

    settings = openmc.Settings()
    settings.run_mode = "eigenvalue"
    settings.source = openmc.IndependentSource(space=openmc.stats.Point())
    settings.particles = int(particles)
    settings.batches = int(batches)
    settings.inactive = int(inactive)
    settings.seed = int(seed)

    energy_filter = openmc.EnergyFilter(GROUP_EDGES)
    flux = openmc.Tally(name="flux")
    flux.filters = [energy_filter]
    flux.scores = ["flux"]

    rates = openmc.Tally(name="rates")
    rates.filters = [energy_filter]
    rates.nuclides = list(N) + ["total"]
    rates.scores = ["absorption", "fission", "nu-fission"]

    scatter = openmc.Tally(name="scatter")
    scatter.filters = [energy_filter, openmc.EnergyoutFilter(GROUP_EDGES)]
    scatter.scores = ["scatter"]

    return openmc.Model(geometry=openmc.Geometry([cell]),
                        materials=openmc.Materials([material]),
                        settings=settings,
                        tallies=openmc.Tallies([flux, rates, scatter]))

In [3]:
def get_results(sp_path):
    """Group constants with index 0 = group 1 (fast), as in FNRP."""
    with openmc.StatePoint(sp_path) as sp:
        keff = sp.keff
        phi = sp.get_tally(name="flux").mean.ravel()[::-1]
        rates = sp.get_tally(name="rates")
        rr = {(nuclide, score): rates.get_values(nuclides=[nuclide], scores=[score]).ravel()[::-1]
              for nuclide in rates.nuclides for score in rates.scores}
        # bins are [g_in, g_out]; transpose so that Sigma_s[g', g] = Sigma_{s, g' <- g}
        scatter = sp.get_tally(name="scatter").mean.reshape(3, 3)[::-1, ::-1].T

    micro = {}
    for nuclide in N:
        fission = rr[nuclide, "fission"]
        micro[nuclide] = {
            "sigma_a": rr[nuclide, "absorption"] / (N[nuclide] * phi),
            "sigma_f": fission / (N[nuclide] * phi),
            "nu": np.divide(rr[nuclide, "nu-fission"], fission,
                            out=np.zeros(3), where=fission > 0),
        }
    return {"keff": keff, "phi": phi,
            "Sigma_a": rr["total", "absorption"] / phi,
            "nuSigma_f": rr["total", "nu-fission"] / phi,
            "Sigma_s": scatter / phi,
            "micro": micro}

## 2. Run the eigenvalue calculation

The medium is infinite and homogeneous, so the starting source position does not
matter, and the fission-source energy distribution settles within a few
generations. A few inactive batches are enough.

In [4]:
model = make_model()
workdir = tempfile.mkdtemp(prefix="hw14_p3_")
print("OpenMC files in", workdir)
sp_path = model.run(cwd=workdir, output=False)
openmc_results = get_results(sp_path)
k = openmc_results["keff"]
print(f"OpenMC k_inf = {k.nominal_value:.5f} +/- {k.std_dev:.5f}")

OpenMC files in /tmp/claude-1000/-home-robertsj-Classes-ne630/bb1933b5-62d9-45f6-8c49-7daeeda903f8/scratchpad/hw14_p3_brtyks36


OpenMC k_inf = 1.13787 +/- 0.00174


## 3. The analytic model from Problems 1 and 2

This reproduces the analytic model with the same FNRP Table 3.2 data, the same
$1/E$ downscatter estimate $\sigma_{s,g+1\leftarrow g}=\sigma_{s,g}\,\xi/\Delta u_g$,
and the same common $\nu=2.43$. Group-2 absorption and fission are resonance
integrals divided by $\Delta u_2=\ln 10^5$. `closure` is Problem 2's
$k_\infty(R)$: downscatter only, with every fission neutron born in group 1.

In [5]:
NU = 2.43  # common fission yield of the Problem 3 analytic check
DU = np.array([np.log(10.0), np.log(1.0e5)])  # lethargy widths of groups 1 and 2
A = {"C12": 12.0, "U235": 235.0, "U238": 238.0}
# FNRP Table 3.2 (barns): g = 1 fission-spectrum average; g = 2 resonance integral
# / ln(1e5), with the thermal scattering value; g = 3 thermal value.
TABLE = {
    "C12": {"sigma_a": [1.23e-3, 1.53e-3 / DU[1], 3.0e-3],
            "sigma_f": [0.0, 0.0, 0.0],
            "sigma_s": [2.36, 4.81, 4.81]},
    "U235": {"sigma_a": [1.29, 404.0 / DU[1], 591.0],
             "sigma_f": [1.22, 272.0 / DU[1], 505.0],
             "sigma_s": [6.33, 15.0, 15.0]},
    "U238": {"sigma_a": [0.361, 278.0 / DU[1], 2.42],
             "sigma_f": [0.304, 2.0e-3 / DU[1], 1.05e-5],
             "sigma_s": [7.42, 9.37, 9.37]},
}


def analytic_constants(nu=NU):
    """Problem 1-2 macroscopic constants in 1/cm, indexed like get_results."""
    Sigma_a, nuSigma_f, Sigma_s = np.zeros(3), np.zeros(3), np.zeros((3, 3))
    micro = {}
    for nuclide, data in TABLE.items():
        alpha = ((A[nuclide] - 1.0) / (A[nuclide] + 1.0))**2
        xi = 1.0 + alpha / (1.0 - alpha) * np.log(alpha)
        sigma_a, sigma_f = np.array(data["sigma_a"]), np.array(data["sigma_f"])
        Sigma_a += N[nuclide] * sigma_a
        nuSigma_f += nu * N[nuclide] * sigma_f
        Sigma_s[1, 0] += N[nuclide] * data["sigma_s"][0] * xi / DU[0]
        Sigma_s[2, 1] += N[nuclide] * data["sigma_s"][1] * xi / DU[1]
        micro[nuclide] = {"sigma_a": sigma_a, "sigma_f": sigma_f,
                          "nu": np.where(sigma_f > 0, nu, 0.0)}
    return {"Sigma_a": Sigma_a, "nuSigma_f": nuSigma_f, "Sigma_s": Sigma_s, "micro": micro}


def closure(c):
    """Problem 2's k_inf: downscatter only, all fission neutrons born in group 1."""
    x = c["Sigma_s"][1, 0] / (c["Sigma_a"][1] + c["Sigma_s"][2, 1])
    y = c["Sigma_s"][2, 1] * x / c["Sigma_a"][2]
    w = np.array([1.0, x, y])
    return x, y, (c["nuSigma_f"] @ w) / (c["Sigma_a"] @ w)


analytic = analytic_constants()
x, y, k_analytic = closure(analytic)
analytic["phi"] = np.array([1.0, x, y])
print(f"analytic: x = {x:.5f}, y = {y:.5f}, k_inf = {k_analytic:.5f}")

analytic: x = 1.78865, y = 6.85861, k_inf = 1.04793


## 4. Comparison

The last line puts OpenMC's group constants into the analytic closure. If it
reproduces OpenMC's $k_\infty$, the three-group balance is adequate, and any
remaining difference from the analytic model comes from the group constants.

In [6]:
x_c, y_c, k_c = closure(openmc_results)
print(f"OpenMC k_inf                            = {k.nominal_value:.5f} +/- {k.std_dev:.5f}")
print(f"analytic k_inf (nu = {NU})               = {k_analytic:.5f}")
print(f"analytic closure with OpenMC constants  = {k_c:.5f}  (x = {x_c:.4f}, y = {y_c:.4f})")

OpenMC k_inf                            = 1.13787 +/- 0.00174
analytic k_inf (nu = 2.43)               = 1.04793
analytic closure with OpenMC constants  = 1.14053  (x = 2.4361, y = 8.5557)


In [7]:
sources = {"analytic": analytic, "OpenMC": openmc_results}
GROUPS = ["g = 1", "g = 2", "g = 3"]
quantities = {
    "φ_g / φ_1": lambda c: c["phi"] / c["phi"][0],
    "Σ_a,g (1/cm)": lambda c: c["Sigma_a"],
    "νΣ_f,g (1/cm)": lambda c: c["nuSigma_f"],
    "Σ_s,g+1←g (1/cm)": lambda c: [c["Sigma_s"][1, 0], c["Sigma_s"][2, 1], np.nan],
}
pd.DataFrame([f(c) for f in quantities.values() for c in sources.values()],
             index=pd.MultiIndex.from_product([quantities, sources]), columns=GROUPS)

g = 1     g = 2    g = 3
φ_g / φ_1        analytic         1     1.789    6.859
                 OpenMC           1     2.504    8.705
Σ_a,g (1/cm)     analytic 0.0001826  0.002779  0.00195
                 OpenMC   5.956e-05  0.001498 0.001806
νΣ_f,g (1/cm)    analytic 8.883e-05 0.0001303 0.002784
                 OpenMC   4.796e-05 0.0001054 0.002519
Σ_s,g+1←g (1/cm) analytic   0.01834  0.007477      NaN
                 OpenMC      0.0191  0.006342      NaN

In [8]:
MICRO = [("C12", "sigma_a"), ("U235", "sigma_a"), ("U235", "sigma_f"), ("U235", "nu"),
         ("U238", "sigma_a"), ("U238", "sigma_f"), ("U238", "nu")]
LABELS = {"sigma_a": "σ_a (b)", "sigma_f": "σ_f (b)", "nu": "ν"}
pd.DataFrame([c["micro"][nuclide][xs] for nuclide, xs in MICRO for c in sources.values()],
             index=pd.MultiIndex.from_tuples([(nuclide, LABELS[xs], name)
                                              for nuclide, xs in MICRO for name in sources]),
             columns=GROUPS)

g = 1     g = 2     g = 3
C12  σ_a (b) analytic   0.00123 0.0001329     0.003
             OpenMC   0.0002666 0.0001012  0.003102
U235 σ_a (b) analytic      1.29     35.09       591
             OpenMC        1.42     30.17     535.2
     σ_f (b) analytic      1.22     23.63       505
             OpenMC       1.251     19.22     456.9
     ν       analytic      2.43      2.43      2.43
             OpenMC       2.546     2.418      2.43
U238 σ_a (b) analytic     0.361     24.15      2.42
             OpenMC      0.2349     12.76     2.161
     σ_f (b) analytic     0.304 0.0001737  1.05e-05
             OpenMC      0.1339 0.0001663 1.489e-05
     ν       analytic      2.43      2.43      2.43
             OpenMC       2.738     2.444     2.443

The analytic model keeps only the $2\leftarrow1$ and $3\leftarrow2$ transfers.
OpenMC's full transfer matrix (row $g'$, column $g$) tests that assumption:

In [9]:
pd.DataFrame(openmc_results["Sigma_s"], index=[f"to g = {g}" for g in (1, 2, 3)],
             columns=[f"from g = {g}" for g in (1, 2, 3)])

,from g = 1,from g = 2,from g = 3
to g = 1,0.3312,0,0
to g = 2,0.0191,0.5311,2.712e-05
to g = 3,0,0.006342,0.548


## 5. Observations

- OpenMC gives $k_\infty = 1.138 \pm 0.002$. The analytic model with $\nu = 2.43$
  gives 1.048, about 8% low.
- The analytic closure with OpenMC's group constants gives 1.141, within 0.3% of
  OpenMC. The three-group balance holds up; the difference is in the group
  constants. The transfer matrix agrees: there is no $3\leftarrow1$ transfer, and
  $2\leftarrow3$ upscatter is about 1.5% of $\Sigma_{a,3}$.
- The largest single difference is U-238 absorption in group 2: 12.8 b, compared
  with 24.1 b. The analytic value is an infinite-dilution resonance integral. At
  $R = 1000$, the large low-energy U-238 resonances are self-shielded. Changing only
  this value in the analytic model raises its $k_\infty$ to 1.195.
- $\Sigma_{s,3\leftarrow2}$ is about 15% below the $1/E$ estimate, which assumes a
  flat flux per unit lethargy across group 2. This and the smaller $\Sigma_{a,2}$
  contribute about equally to OpenMC's larger $\phi_2/\phi_1$.
- The group-3 values are averages over the actual spectrum below 1 eV rather than
  0.0253 eV point values. U-235 and U-238 come out about 10% lower, while C-12 does
  not. This lowers the thermal ratio $\nu\Sigma_{f,3}/\Sigma_{a,3}$ from 1.43 to
  1.39.
- OpenMC's group 1 extends down to 0.1 MeV, where the slowing-down spectrum is softer than
  a fission spectrum, so U-238 fast fission is 0.13 b rather than 0.30 b. Group 1
  produces only about 0.2% of the fission neutrons, so this barely affects $k_\infty$.